In [ ]:
import pandas as pd

df = pd.read_csv("/IMDB Dataset.csv")

In [ ]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
df.shape

(50000, 2)

In [ ]:
df.isnull().sum()

,0
review,0
sentiment,0


In [ ]:
df.drop_duplicates(inplace =True)

In [ ]:
df.shape

(49582, 2)

# Pre-processing

## Coverting to lower case

In [ ]:
df["review"] = df["review"].str.lower()

## Remove Url

In [ ]:
import re
def remove_urls(text):
    test = re.sub(r"http\S+" ,"" ,text)
    return text

df["review"] = df["review"].apply(remove_urls)

## Remove Punctuation

In [ ]:
def remove_puncutuation(text):
    text = re.sub(r"[^A-Za-z0-9\S]" ,"" ,text)
    return text
df["review"] = df["review"].apply(remove_puncutuation)

## Remove HTML

In [ ]:
def remove_html(text):
    text = re.sub(r"<.?>" , "" , text)
    return text
df["review"] = df["review"].apply(remove_html)

## Remove Stopword

In [ ]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text

df["review"] = df["review"].apply(remove_stopwords)


In [ ]:
df.head()

,review,sentiment
0,oneoftheotherreviewershasmentionedthatafterwat...,positive
1,awonderfullittleproduction.<br/><br/>thefilmin...,positive
2,ithoughtthiswasawonderfulwaytospendtimeonatooh...,positive
3,basicallythere'safamilywherealittleboy(jake)th...,negative
4,"pettermattei's""loveinthetimeofmoney""isavisuall...",positive


## Stemming

In [ ]:

from nltk.stem import PorterStemmer


def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [ ]:
df.head()

,review,sentiment
0,oneoftheotherreviewershasmentionedthatafterwat...,positive
1,awonderfullittleproduction. < br/ > < br/ > th...,positive
2,ithoughtthiswasawonderfulwaytospendtimeonatooh...,positive
3,basicallythere'safamilywherealittleboy ( jake ...,negative
4,pettermattei 's '' loveinthetimeofmoney '' isa...,positive


## Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [ ]:
y = df["sentiment"]

In [ ]:
y

,sentiment
0,1
1,1
2,1
3,0
4,1
...,...
49995,1
49996,0
49997,0
49998,0


## Vectorization

In [ ]:
df.head()

,review,sentiment
0,oneoftheotherreviewershasmentionedthatafterwat...,1
1,awonderfullittleproduction. < br/ > < br/ > th...,1
2,ithoughtthiswasawonderfulwaytospendtimeonatooh...,1
3,basicallythere'safamilywherealittleboy ( jake ...,0
4,pettermattei 's '' loveinthetimeofmoney '' isa...,1


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

# Datasets and Dataloaders

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train ,X_test , y_train , y_test = train_test_split(
    X , y , test_size=0.2 , random_state=42
    )

In [ ]:
X_train.shape

(39665, 5000)

In [ ]:
X_test.shape

(9917, 5000)

In [ ]:
import torch

In [ ]:
from torch.utils.data import TensorDataset , DataLoader

In [ ]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [ ]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [ ]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

## Build RNN

In [ ]:
import torch.nn as nn
import torch.optim as optim

In [ ]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0)
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [ ]:
# Create model
input_size = X_train.shape[1]
hidden_size = 64
model = RNN(input_size,hidden_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Training the RNN

In [ ]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction

        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.5977111458778381
epoch = 2/10 and loss = 0.5559033155441284
epoch = 3/10 and loss = 0.6049597859382629
epoch = 4/10 and loss = 0.507097065448761
epoch = 5/10 and loss = 0.41117480397224426
epoch = 6/10 and loss = 0.5853294134140015
epoch = 7/10 and loss = 0.510187566280365
epoch = 8/10 and loss = 0.5450464487075806
epoch = 9/10 and loss = 0.5052241683006287
epoch = 10/10 and loss = 0.5200361013412476


## Evalution

In [ ]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 66.91539780175457
